In [2]:
from google.colab import drive
import pandas as pd
import numpy as np

drive.mount('/content/drive')
carpeta = '/content/drive/MyDrive/ACDSA_2027/'

d1 = pd.read_csv(carpeta + 'Atrato_D1_PScore_v2.csv')
d2 = pd.read_csv(carpeta + 'Atrato_D2_NDTI_consolidado.csv')
d4 = pd.read_csv(carpeta + 'Atrato_D4_Hansen.csv')
d5 = pd.read_csv(carpeta + 'Atrato_D5_MapBiomas.csv')

df = d1[['HYBAS_ID', 'D1_PScore']].rename(columns={'D1_PScore': 'D1_raw'})
df = df.merge(d2[['HYBAS_ID', 'D2_NDTI']].rename(columns={'D2_NDTI': 'D2_raw'}), on='HYBAS_ID', how='outer')
df = df.merge(d4[['HYBAS_ID', 'D4_perdida_ha']].rename(columns={'D4_perdida_ha': 'D4_raw'}), on='HYBAS_ID', how='outer')
df = df.merge(d5[['HYBAS_ID', 'D5_mineria_ha']].rename(columns={'D5_mineria_ha': 'D5_raw'}), on='HYBAS_ID', how='outer')
df = df.merge(d1[['HYBAS_ID', 'area_total_ha']], on='HYBAS_ID', how='left')

df['n_indicadores_validos'] = df[['D1_raw', 'D2_raw', 'D4_raw', 'D5_raw']].notna().sum(axis=1)

def norm_min_max(serie):
    mn, mx = serie.min(), serie.max()
    if mx == mn:
        return pd.Series([0.5] * len(serie), index=serie.index)
    return (serie - mn) / (mx - mn)

# CAMBIO CAMINO B: log(1+x) para D4 y D5 antes de min-max
df['D1_norm'] = 1 - norm_min_max(-df['D1_raw'])
df['D2_norm'] = 1 - norm_min_max(df['D2_raw'])
df['D4_norm'] = 1 - norm_min_max(np.log1p(df['D4_raw']))
df['D5_norm'] = 1 - norm_min_max(np.log1p(df['D5_raw']))

print("=" * 60)
print("NORMALIZACIÓN CON TRANSFORMACIÓN LOG PARA D4 Y D5")
print("=" * 60)
for col in ['D1_norm', 'D2_norm', 'D4_norm', 'D5_norm']:
    print(f"  {col}: min={df[col].min():.3f}, max={df[col].max():.3f}, media={df[col].mean():.3f}")

df['ICSHM_EO'] = df[['D1_norm', 'D2_norm', 'D4_norm', 'D5_norm']].mean(axis=1, skipna=True)
df.loc[df['n_indicadores_validos'] < 2, 'ICSHM_EO'] = np.nan

tau = 0.20
def aplicar_veto(row):
    if pd.isna(row['ICSHM_EO']):
        return np.nan
    vals = [row['D1_norm'], row['D2_norm'], row['D4_norm'], row['D5_norm']]
    vals = [v for v in vals if not pd.isna(v)]
    return any(v < tau for v in vals) if vals else False

df['veto_activado'] = df.apply(aplicar_veto, axis=1)

def clasificar(row):
    if pd.isna(row['ICSHM_EO']): return 'Sin dato'
    if row['veto_activado']: return 'Crítico (veto)'
    if row['ICSHM_EO'] < 0.20: return 'Crítico'
    elif row['ICSHM_EO'] < 0.40: return 'Bajo'
    elif row['ICSHM_EO'] < 0.60: return 'Medio'
    elif row['ICSHM_EO'] < 0.80: return 'Alto'
    else: return 'Seguro'

df['categoria'] = df.apply(clasificar, axis=1)

print("\n" + "=" * 60)
print("ESTADÍSTICAS ICSHM-EO CON CAMINO B")
print("=" * 60)
print(df['ICSHM_EO'].describe())

print("\nDistribución por categoría:")
print(df['categoria'].value_counts())

print(f"\nVeto activado: {df['veto_activado'].sum()} subcuencas")

print("\nTOP 10 MÁS CRÍTICAS:")
print(df.sort_values('ICSHM_EO').head(10)[['HYBAS_ID', 'ICSHM_EO', 'categoria']].to_string(index=False))

print("\nTOP 10 MÁS SEGURAS:")
print(df.sort_values('ICSHM_EO', ascending=False).head(10)[['HYBAS_ID', 'ICSHM_EO', 'categoria']].to_string(index=False))

print("\nCORRELACIONES:")
print(df[['D1_norm', 'D2_norm', 'D4_norm', 'D5_norm']].corr().round(3))

df.to_csv(carpeta + 'Atrato_ICSHM_EO_tabla_maestra.csv', index=False)
print("\n✅ Guardado")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NORMALIZACIÓN CON TRANSFORMACIÓN LOG PARA D4 Y D5
  D1_norm: min=0.000, max=1.000, media=0.482
  D2_norm: min=0.000, max=1.000, media=0.477
  D4_norm: min=0.000, max=1.000, media=0.550
  D5_norm: min=0.000, max=1.000, media=0.907

ESTADÍSTICAS ICSHM-EO CON CAMINO B
count    170.000000
mean       0.615613
std        0.124666
min        0.347273
25%        0.507762
50%        0.620666
75%        0.693452
max        0.992448
Name: ICSHM_EO, dtype: float64

Distribución por categoría:
categoria
Alto              80
Crítico (veto)    40
Medio             37
Seguro            12
Bajo               1
Name: count, dtype: int64

Veto activado: 40 subcuencas

TOP 10 MÁS CRÍTICAS:
  HYBAS_ID  ICSHM_EO      categoria
6090106880  0.347273           Bajo
6090108730  0.373480 Crítico (veto)
6090112900  0.377629 Crítico (veto)
6090110460  0.382627 Crítico (veto)
6090107630  